# Maven Market Data Cleaning

This notebook applies the cleaning rules agreed after the data audit. It keeps the raw files unchanged, preserves all uncertain duplicate-looking rows, and writes reproducible outputs to `data/processed/`.

## Goal

Prepare the seven analytical tables for modeling and Power BI without inventing values or silently removing records.

The main decisions are:

- combine the 1997 and 1998 transaction files;
- convert dates to real date types;
- keep account numbers, postal codes, SKUs, phone numbers, and row IDs as identifiers;
- convert source flags to readable boolean fields;
- add traceable row IDs where the source has no identifier;
- flag duplicate candidates instead of deleting them.

## Setup

Run the notebook from the repository root with the eight Maven Market files available in `data/raw/`. The cleaning logic lives in `scripts/data_cleaning.py`, so the notebook and command-line workflow use the same code.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from data_cleaning import run_cleaning, OUTPUT_FILES

pd.set_option("display.max_colwidth", 90)
print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/maven-market-retail-intelligence-phase-04-data-cleaning


## Steps

### 1. Run the cleaning pipeline

The function loads the raw files, applies the transformations, writes the processed CSVs, and runs validation checks before reporting success.

In [2]:
raw_tables, cleaned_tables, change_log, before_after, validation = run_cleaning()
print(f"Validation status: {validation['validation_status']}")

Validation status: passed


### 2. Compare the data before and after cleaning

Row counts should stay unchanged. The additional columns in transactions and returns provide lineage, surrogate row IDs, and duplicate-candidate flags.

In [3]:
before_after

,table,rows_before,rows_after,columns_before,columns_after,missing_cells_before,missing_cells_after,exact_duplicate_rows_before,duplicate_candidate_rows_after
0,calendar,730,730,1,1,0,0,0,0
1,customers,10281,10281,20,21,1,0,0,0
2,products,1560,1560,9,9,1695,0,0,0
3,regions,109,109,3,3,0,0,0,0
4,returns,7087,7087,4,7,0,0,5,10
5,stores,24,24,13,13,0,0,0,0
6,transactions,269720,269720,6,10,0,0,9,18


### 3. Review the changes

The log records what changed, how many rows were affected, and why the change was made.

In [4]:
change_log

,table,change,affected_rows,reason
0,transactions,Appended 1997 and 1998 files,269720,Create one consistent sales table while preserving source-year lineage.
1,transactions,Added surrogate line ID and source row fields,269720,The source has no transaction or order identifier.
2,transactions,Flagged exact-looking duplicate candidates,18,Retain uncertain rows but make them visible for analysis and review.
3,returns,Added surrogate line ID and source row number,7087,The source has no return-row identifier.
4,returns,Flagged exact-looking duplicate candidates,10,"Identical returns may still be separate real events, so they were not removed."
5,customers,Converted account and postal identifiers to text,10281,Identifiers should not be aggregated as numbers.
6,customers,Restored five-character postal-code display,20,Four-digit source values had lost a leading zero.
7,customers,Filled missing display surname and kept a missing-value flag,1,Avoid blank labels without hiding that the source value was missing.
8,customers,Decoded compact demographic codes,10281,Readable labels are safer for reports and Power BI fields.
9,products,Converted SKU to text,1560,"SKU is an identifier, not a numeric measure."


### 4. Inspect key transformations

In [5]:
cleaned_tables["customers"].loc[
    cleaned_tables["customers"]["last_name_missing"],
    ["customer_id", "first_name", "last_name", "last_name_missing", "customer_postal_code"]
]

,customer_id,first_name,last_name,last_name_missing,customer_postal_code
5139,5140,Robert,Unknown,True,55238


In [6]:
cleaned_tables["products"][[
    "product_id", "product_sku", "is_recyclable", "is_low_fat"
]].head()

,product_id,product_sku,is_recyclable,is_low_fat
0,1,90748583674,False,False
1,2,96516502499,False,True
2,3,58427771925,True,True
3,4,64412155747,True,False
4,5,85561191439,True,False


In [7]:
cleaned_tables["transactions"].loc[
    cleaned_tables["transactions"]["duplicate_candidate"]
].head(10)

,transaction_line_id,transaction_date,stock_date,product_id,customer_id,store_id,quantity,source_year,source_row_number,duplicate_candidate
28560,TX-1997-028561,1997-05-02,1997-05-01,290,1690,16,3,1997,28561,True
28638,TX-1997-028639,1997-05-02,1997-05-01,290,1690,16,3,1997,28639,True
144309,TX-1998-057473,1998-04-29,1998-04-22,468,9610,8,3,1998,57473,True
144310,TX-1998-057474,1998-04-29,1998-04-22,468,9610,8,3,1998,57474,True
154802,TX-1998-067966,1998-05-19,1998-05-15,784,6527,12,5,1998,67966,True
154803,TX-1998-067967,1998-05-19,1998-05-15,784,6527,12,5,1998,67967,True
157000,TX-1998-070164,1998-05-24,1998-05-19,426,77,7,4,1998,70164,True
157001,TX-1998-070165,1998-05-24,1998-05-19,426,77,7,4,1998,70165,True
168747,TX-1998-081911,1998-06-18,1998-06-11,1528,4731,18,3,1998,81911,True
168748,TX-1998-081912,1998-06-18,1998-06-11,1528,4731,18,3,1998,81912,True


## Checks

These checks confirm that row counts were preserved, generated IDs are unique, foreign keys still match their dimensions, identifier columns have the intended in-memory types, and no invalid quantities or date-ordering issues were introduced.

In [8]:
validation_table = pd.DataFrame(
    [{"check": key, "result": value} for key, value in validation.items()]
)
validation_table

,check,result
0,transaction_rows_preserved,True
1,return_rows_preserved,True
2,customer_rows_preserved,True
3,product_rows_preserved,True
4,store_rows_preserved,True
5,region_rows_preserved,True
6,calendar_rows_preserved,True
7,clean_missing_cells,0
8,transaction_id_unique,True
9,return_id_unique,True


### Output files

In [9]:
output_rows = []
for table, filename in OUTPUT_FILES.items():
    path = PROJECT_ROOT / "data" / "processed" / filename
    output_rows.append({
        "table": table,
        "file": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(cleaned_tables[table]),
        "size_mb": round(path.stat().st_size / 1_000_000, 3),
    })
pd.DataFrame(output_rows)

,table,file,rows,size_mb
0,calendar,data/processed/calendar_clean.csv,730,0.008
1,customers,data/processed/customers_clean.csv,10281,1.783
2,products,data/processed/products_clean.csv,1560,0.116
3,regions,data/processed/regions_clean.csv,109,0.003
4,returns,data/processed/returns_clean.csv,7087,0.297
5,stores,data/processed/stores_clean.csv,24,0.003
6,transactions,data/processed/transactions_clean.csv,269720,18.377


## Takeaways

- No transaction, return, customer, product, store, region, or calendar rows were removed.
- The two transaction files now form one table with 269,720 rows.
- Eighteen transaction rows and ten return rows are marked as duplicate candidates, but remain in the data.
- Postal-code display is repaired for 20 customers.
- The one missing surname is shown as `Unknown`, with a separate flag preserving the original missingness.
- Product packaging flags are now explicit booleans rather than blank/1 values.
- The cleaned tables passed the integrity and business-rule checks.

The next step is dimensional modeling. Calculated revenue, cost, profit, and date attributes will be introduced there rather than mixed into the cleaning layer.